# **SET UP**

**Import neccessary libraries.** Also make sure to **upload the SQLite file** to the runtime, this is necessary for compiling

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets  # Colab ships with it
from IPython.display import display

# For a nicer inline plotting style:
%matplotlib inline


**Load in the data** from the larger data set

In [ ]:
# Open a connection to the SQLite file
conn = sqlite3.connect("database.sqlite")
cursor = conn.cursor()

# List all tables to verify
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Found these tables in database.sqlite:\n")
for t in tables:
    print("  -", t[0])


Found these tables in database.sqlite:

  - sqlite_sequence
  - Player_Attributes
  - Player
  - Match
  - League
  - Country
  - Team
  - Team_Attributes


**Get the ID** of FC Barcelona so we can filter the data

In [ ]:
# Fetch Barcelona’s team_api_id to use in table creation
cursor.execute("""
    SELECT team_api_id
    FROM Team
    WHERE team_long_name = 'FC Barcelona'
""")
barca_id_row = cursor.fetchone()
if barca_id_row is None:
    raise ValueError("Couldn’t find 'FC Barcelona' in Team.team_long_name")
barca_id = barca_id_row[0]

print("FC Barcelona’s team_api_id =", barca_id)


FC Barcelona’s team_api_id = 8634


# **TABLE CREATION**

Table for **the matches Barcelona's played in**

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_matches;")

# Create barca_matches by filtering if Barca was either home or away team:
cursor.execute(f"""
    CREATE TABLE barca_matches AS
    SELECT *
    FROM Match
    WHERE home_team_api_id = {barca_id}
       OR away_team_api_id = {barca_id}
""")
conn.commit()

# See how many matches were populated (quick check)
count_row = cursor.execute("SELECT COUNT(*) FROM barca_matches;").fetchone()
print(f"Number of rows in barca_matches: {count_row[0]}")


Number of rows in barca_matches: 304


Table for **the leagues Barcelona's played in**

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_league;")

# Create barca_league by grabbing any League that shows up in barca_matches
cursor.execute("""
    CREATE TABLE barca_league AS
    SELECT DISTINCT l.*
    FROM League AS l
    JOIN (
      SELECT league_id
      FROM barca_matches
    ) AS sub_league
    ON l.id = sub_league.league_id;
""")
conn.commit()

# Quick sanity check
print("→ Leagues in barca_league:")
print(
    pd.read_sql_query(
        "SELECT id, name FROM barca_league ORDER BY name;",
        conn
    )
)

→ Leagues in barca_league:
      id             name
0  21518  Spain LIGA BBVA


Table for **teams** (mapping table for either the Barca team or their opponent)

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_team;")

# Create barca_team by grabbing any Team that shows up (as home or away) in barca_matches
cursor.execute("""
    CREATE TABLE barca_team AS
    SELECT DISTINCT t.*
    FROM Team AS t
    JOIN (
      SELECT home_team_api_id AS tid FROM barca_matches
      UNION
      SELECT away_team_api_id AS tid FROM barca_matches
    ) AS sub_teams
    ON t.team_api_id = sub_teams.tid;
""")
conn.commit()

# Quick sanity check
n = cursor.execute("SELECT COUNT(*) FROM barca_team;").fetchone()[0]
print(f"→ Number of rows in barca_team: {n}")
print(
    pd.read_sql_query(
        "SELECT team_api_id, team_long_name FROM barca_team ORDER BY team_long_name LIMIT 10;",
        conn
    )
)


→ Number of rows in barca_team: 33
   team_api_id           team_long_name
0         8315  Athletic Club de Bilbao
1         9906          Atlético Madrid
2         8371               CA Osasuna
3         8388              CD Numancia
4         9867              CD Tenerife
5         7869               Córdoba CF
6        10268                 Elche CF
7         8634             FC Barcelona
8         8305                Getafe CF
9         7878               Granada CF


Table for the **stats of a team** in "barca_team"

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_team_stats;")

# Create barca_team_attributes by filtering Team_Attributes to only those team_api_id in barca_team
cursor.execute("""
    CREATE TABLE barca_team_stats AS
    SELECT ta.*
    FROM Team_Attributes AS ta
    WHERE ta.team_api_id IN (
      SELECT team_api_id
      FROM barca_team
    );
""")
conn.commit()

# Sanity check: show a few columns for Barca’s stats
print("→ Sample of barca_team_stats:")
print(
    pd.read_sql_query(
        "SELECT team_api_id, date, buildUpPlaySpeed, chanceCreationPassing FROM barca_team_stats LIMIT 5;",
        conn
    )
)


→ Sample of barca_team_stats:
   team_api_id                 date  buildUpPlaySpeed  chanceCreationPassing
0         9865  2010-02-22 00:00:00                65                     45
1         9865  2011-02-22 00:00:00                55                     57
2         9865  2012-02-22 00:00:00                42                     57
3         9865  2013-09-20 00:00:00                46                     57
4         9865  2014-09-19 00:00:00                46                     57


Table for the **players on the Barca squad** (mapping table for attributes)

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_player;")

# Create barca_player by selecting only Barca’s own roster columns
cursor.execute(f"""
    CREATE TABLE barca_player AS
    SELECT DISTINCT p.*
    FROM Player AS p
    JOIN (
      -- If Barça was home, pull home_player_1..home_player_11
      SELECT home_player_1 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_2 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_3 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_4 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_5 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_6 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_7 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_8 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_9 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_10 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}
      UNION
      SELECT home_player_11 AS pid FROM barca_matches WHERE home_team_api_id = {barca_id}

      UNION

      -- If Barça was away, pull away_player_1..away_player_11
      SELECT away_player_1 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_2 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_3 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_4 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_5 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_6 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_7 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_8 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_9 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_10 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
      UNION
      SELECT away_player_11 AS pid FROM barca_matches WHERE away_team_api_id = {barca_id}
    ) AS only_barca
    ON p.player_api_id = only_barca.pid
""")
conn.commit()

# See how many Barca players were populated (quick check)
count_row = cursor.execute("SELECT COUNT(*) FROM barca_player;").fetchone()
print(f"Number of rows in barca_player: {count_row[0]}")

# Preview a few names
print(
    pd.read_sql_query(
        "SELECT player_api_id, player_name FROM barca_player ORDER BY player_name LIMIT 10;",
        conn
    )
)


Number of rows in barca_player: 57
   player_api_id     player_name
0          33992         Adriano
1         189192     Aleix Vidal
2          26169  Aleksandr Hleb
3          23678       Alex Song
4          50047  Alexis Sanchez
5          30955  Andres Iniesta
6         172948   Andreu Fontas
7          39792      Arda Turan
8          96643     Bojan Krkic
9          30661    Carles Puyol


Table for the **stats of the players**

In [ ]:
# Drop table if it already exists
cursor.execute("DROP TABLE IF EXISTS barca_player_stats;")

# Create barca_player_stats by filtering Player_Attributes for only those player_api_id in barca_player
cursor.execute("""
    CREATE TABLE barca_player_stats AS
    SELECT pa.*
    FROM Player_Attributes AS pa
    WHERE pa.player_api_id IN (
      SELECT player_api_id
      FROM barca_player
    );
""")
conn.commit()

# Sanity check: show a few columns for Barca players (first 5 rows)
print("→ Sample of barca_player_stats:")
print(
    pd.read_sql_query(
        "SELECT player_api_id, date, overall_rating, potential FROM barca_player_stats LIMIT 5;",
        conn
    )
)


→ Sample of barca_player_stats:
   player_api_id                 date  overall_rating  potential
0          33992  2016-05-12 00:00:00              77         77
1          33992  2016-04-21 00:00:00              77         77
2          33992  2016-03-24 00:00:00              79         79
3          33992  2015-10-16 00:00:00              79         79
4          33992  2015-09-21 00:00:00              79         79


# **INDEXES**

**Create indexes** to allow faster querying

In [ ]:
# 1) INDEXES FOR barca_matches
# -----------------------------------

# Drop existing index on home_team_api_id if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_matches_home;")
# Create index on home_team_api_id for efficient home‐team lookups
cursor.execute("CREATE INDEX idx_barca_matches_home ON barca_matches(home_team_api_id);")

# Drop existing index on away_team_api_id if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_matches_away;")
# Create index on away_team_api_id for efficient away‐team lookups
cursor.execute("CREATE INDEX idx_barca_matches_away ON barca_matches(away_team_api_id);")

# Drop existing index on date if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_matches_date;")
# Create index on date to speed up any time‐range or season filtering
cursor.execute("CREATE INDEX idx_barca_matches_date ON barca_matches(date);")

# Drop existing index on season if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_matches_season;")
# Create index on season to speed up grouping/aggregations by season
cursor.execute("CREATE INDEX idx_barca_matches_season ON barca_matches(season);")

# Commit after creating indexes on barca_matches
conn.commit()


# 2) INDEXES FOR barca_player_attributes
# -----------------------------------------

# Drop existing composite index on (player_api_id, date) if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_player_attr_pid_date;")
# Create composite index on (player_api_id, date) for quick filtering by player and season‐date range
cursor.execute("""
    CREATE INDEX idx_barca_player_attr_pid_date
    ON barca_player_stats(player_api_id, date);
""")

# Commit after creating index on barca_player_attributes
conn.commit()


# 3) INDEX FOR barca_player
# ------------------------------

# Drop existing index if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_player_pid;")
# Create index on player_api_id so joins from barca_matches → barca_player are fast
cursor.execute("CREATE INDEX idx_barca_player_pid ON barca_player(player_api_id);")

conn.commit()


# 4) INDEX FOR barca_team
# ----------------------------

# Drop existing index if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_team_tid;")
# Create index on team_api_id so joins from barca_matches → barca_team are fast
cursor.execute("CREATE INDEX idx_barca_team_tid ON barca_team(team_api_id);")

conn.commit()


# 5) INDEXES FOR barca_team_attributes
# ----------------------------------------

# Drop existing composite index on (team_api_id, date) if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_team_attr_tid_date;")
# Create composite index on (team_api_id, date) for filtering team attributes by date ranges
cursor.execute("""
    CREATE INDEX idx_barca_team_attr_tid_date
    ON barca_team_stats(team_api_id, date);
""")

conn.commit()


# 6) INDEX FOR barca_league
# ----------------------------

# Drop existing index if it exists
cursor.execute("DROP INDEX IF EXISTS idx_barca_league_id;")
# Create index on id (league_id) to speed up joins from barca_matches → barca_league
cursor.execute("CREATE INDEX idx_barca_league_id ON barca_league(id);")

conn.commit()


# 7) VERIFICATION OUTPUT
# -------------------------------------

print("Indexes successfully created:")
for row in cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND name LIKE 'idx_barca_%';"):
    print("  -", row[0])

# ------------------------------------------------------------------
# 8) EXTRA INDEXES FOR VISUALIZATION QUERIES
# ------------------------------------------------------------------

# -- 8a) Case-insensitive name look-up on barca_player
cursor.execute("DROP INDEX IF EXISTS idx_barca_player_name_lower;")
cursor.execute("""
    CREATE INDEX idx_barca_player_name_lower
    ON barca_player (lower(player_name));
""")

# -- 8b) Fast player_id look-up on barca_player_stats
cursor.execute("DROP INDEX IF EXISTS idx_barca_player_stats_pid;")
cursor.execute("""
    CREATE INDEX idx_barca_player_stats_pid
    ON barca_player_stats (player_api_id);
""")

# -- 8c) Opponent look-up on barca_matches  (skip if column absent)
import sqlite3
try:
    cursor.execute("DROP INDEX IF EXISTS idx_barca_matches_opponent;")
    cursor.execute("""
        CREATE INDEX idx_barca_matches_opponent
        ON barca_matches (opponent_team_api_id);
    """)
except sqlite3.OperationalError:
    # barca_matches may not have opponent_team_api_id – ignore gracefully
    pass

conn.commit()


# ------------------------------------------------------------------
# 9) SEASON HELPER VIEW  (shifts Jan–Jun back to previous season)
# ------------------------------------------------------------------
cursor.execute("DROP VIEW IF EXISTS v_match_season;")
cursor.execute("""
    CREATE VIEW v_match_season AS
    SELECT
        *,
        /* e.g., a match on 2009-02-14 counts toward season '2008' */
        strftime('%Y', date(date, '-6 months')) AS season
    FROM barca_matches;
""")

conn.commit()


# ------------------------------------------------------------------
# 10) QUICK CHECK
# ------------------------------------------------------------------
print("\nAdditional indexes now present:")
for row in cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='index'
      AND name IN (
          'idx_barca_player_name_lower',
          'idx_barca_player_stats_pid',
          'idx_barca_matches_opponent'
      );
"""):
    print("  •", row[0])

print("\nView 'v_match_season' created successfully.")




Indexes successfully created:
  - idx_barca_matches_home
  - idx_barca_matches_away
  - idx_barca_matches_date
  - idx_barca_matches_season
  - idx_barca_player_attr_pid_date
  - idx_barca_player_pid
  - idx_barca_team_tid
  - idx_barca_team_attr_tid_date
  - idx_barca_league_id

Additional indexes now present:
  • idx_barca_player_name_lower
  • idx_barca_player_stats_pid

View 'v_match_season' created successfully.


# **VISUALIZATIONS**

***Run the following query to see which player's you can use in the next 2 queries!!***

In [ ]:
# Query to count how many Barça matches each player appeared in, ordered descending.
player_appearances_query = f"""
SELECT
  p.player_name,
  COUNT(*) AS appearances
FROM barca_player AS p
JOIN barca_matches AS m
  ON (
       (m.home_team_api_id = {barca_id} AND (
         p.player_api_id = m.home_player_1 OR
         p.player_api_id = m.home_player_2 OR
         p.player_api_id = m.home_player_3 OR
         p.player_api_id = m.home_player_4 OR
         p.player_api_id = m.home_player_5 OR
         p.player_api_id = m.home_player_6 OR
         p.player_api_id = m.home_player_7 OR
         p.player_api_id = m.home_player_8 OR
         p.player_api_id = m.home_player_9 OR
         p.player_api_id = m.home_player_10 OR
         p.player_api_id = m.home_player_11
       ))
    OR (m.away_team_api_id = {barca_id} AND (
         p.player_api_id = m.away_player_1 OR
         p.player_api_id = m.away_player_2 OR
         p.player_api_id = m.away_player_3 OR
         p.player_api_id = m.away_player_4 OR
         p.player_api_id = m.away_player_5 OR
         p.player_api_id = m.away_player_6 OR
         p.player_api_id = m.away_player_7 OR
         p.player_api_id = m.away_player_8 OR
         p.player_api_id = m.away_player_9 OR
         p.player_api_id = m.away_player_10 OR
         p.player_api_id = m.away_player_11
       ))
     )
GROUP BY p.player_name
ORDER BY appearances DESC
LIMIT 20;
"""

# Execute and display
players_df = pd.read_sql_query(player_appearances_query, conn)
players_df


,player_name,appearances
0,Lionel Messi,249
1,Daniel Alves,225
2,Sergio Busquets,213
3,Gerard Pique,209
4,Victor Valdes,197
5,Andres Iniesta,190
6,Xavi Hernandez,189
7,Javier Mascherano,153
8,Pedro Rodriguez,136
9,Carles Puyol,114


**Player Visualizations**

View a player's rating throughout a season. Input a year between 2008-2016 and a player's first name from the above list.

Enter a player and a year, **find their best rating for a given season.**

In [ ]:
# ------------------------------------------------------------------
# SQL helpers
# ------------------------------------------------------------------
PLAYER_SQL = """
SELECT player_api_id, player_name
FROM barca_player
WHERE lower(player_name) LIKE '%' || lower(?) || '%'
ORDER BY player_name
"""

SNAPSHOT_SQL = """
SELECT
    ps.overall_rating,
    ps.finishing,
    ps.short_passing,
    ps.acceleration,
    ps.strength,
    ps.date
FROM barca_player_stats ps
WHERE ps.player_api_id = ?
  AND strftime('%Y', date(ps.date, '-6 months')) = ?  -- season mapping
  AND ps.overall_rating IS NOT NULL
ORDER BY ps.overall_rating DESC, ps.date DESC
LIMIT 1;
"""

ATTRS  = ["overall_rating", "finishing", "short_passing", "acceleration", "strength"]
LABELS = ["Overall", "Finishing", "Passing", "Acceleration", "Strength"]

# ------------------------------------------------------------------
# Radar‑plot helper
# ------------------------------------------------------------------

def _plot_radar(scores, *, title: str = "") -> None:
    import numpy as np
    angles = np.linspace(0, 2 * np.pi, len(scores), endpoint=False).tolist()
    scores_loop = scores + scores[:1]
    angles_loop = angles + angles[:1]

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, polar=True)
    ax.plot(angles_loop, scores_loop, linewidth=2)
    ax.fill(angles_loop, scores_loop, alpha=0.30)

    ax.set_thetagrids(np.degrees(angles), LABELS)
    ax.set_title(title, size=14, pad=20)
    ax.grid(True)
    plt.show()

# ------------------------------------------------------------------
# DB helper
# ------------------------------------------------------------------

def _fetch_best_snapshot(player_id: int, season: str):
    row = conn.execute(SNAPSHOT_SQL, (player_id, season)).fetchone()
    if row is None:
        return None
    return dict(zip(ATTRS + ["date"], row))

# ------------------------------------------------------------------
# UI callbacks
# ------------------------------------------------------------------

def _on_submit(_):
    from IPython.display import clear_output

    clear_output(wait=True)
    season_year = year_widget.value
    last_name   = name_widget.value.strip()
    if not last_name:
        print("Please enter a player last name.")
        return

    season = str(season_year)

    candidates = conn.execute(PLAYER_SQL, (last_name,)).fetchall()
    if not candidates:
        print(f"No player found matching '{last_name}'. Try another name.")
        return

    player_id, player_name = candidates[0]

    snap = _fetch_best_snapshot(player_id, season)
    if snap is None:
        nxt = int(season) + 1
        print(f"No data for {player_name} in season {season}/{nxt}.")
        return

    nxt = int(season) + 1
    title = f"{player_name} – Best attributes in {season}/{nxt}\n({snap['date']})"
    scores = [snap[attr] for attr in ATTRS]
    _plot_radar(scores, title=title)

# ------------------------------------------------------------------
# Public entry point
# ------------------------------------------------------------------

def show_player_season_snapshot():
    """Display interactive widget for Player Visual 1."""
    global year_widget, name_widget

    import ipywidgets as widgets

    year_widget = widgets.BoundedIntText(
        value=2008,
        min=1996,
        max=2025,
        step=1,
        description='Year:',
        layout=widgets.Layout(width='180px')  # widened input box
    )
    name_widget = widgets.Text(
        value='',
        placeholder='last name',
        description='Player:',
        layout=widgets.Layout(width='220px')
    )
    go_button = widgets.Button(description='Show')
    go_button.on_click(_on_submit)

    from IPython.display import display
    display(widgets.HBox([year_widget, name_widget, go_button]))

# ------------------------------------------------------------------
# Auto‑render
# ------------------------------------------------------------------
show_player_season_snapshot()


For a given player, **show some statistics accross their matches.**

In [ ]:
# -------------------------------------------------------------------
# Player Visual 2 – “cool stats”  (robust version)
# -------------------------------------------------------------------
import sqlite3, ipywidgets as widgets, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

# ---------- helper: lineup columns ---------------------------------
def _lineup_cols():
    cols = [r[1] for r in conn.execute("PRAGMA table_info(barca_matches);")]
    return [c for c in cols if "player" in c.lower()]

LINEUP_COLS = _lineup_cols()

# ---------- peak overall_rating query ------------------------------
PEAK_SQL = """
SELECT overall_rating, date
FROM   barca_player_stats
WHERE  player_api_id = ?
ORDER BY overall_rating DESC, date ASC
LIMIT 1;
"""

# -------------------------------------------------------------------
def show_player_cool_stats(last_name: str):
    last_name = last_name.strip().lower()
    if not last_name:
        print("Enter a last name.")
        return

    # --- resolve player -------------------------------------------------
    row = conn.execute(
        "SELECT player_api_id, player_name FROM barca_player "
        "WHERE lower(player_name) LIKE ? LIMIT 1",
        (f"%{last_name}%",)
    ).fetchone()
    if row is None:
        print(f"No Barça player found matching '{last_name}'.")
        return

    player_id, player_name = row
    print(f"↳  Found {player_name} (id {player_id})")

    # --- pull matches where that player appears -------------------------
    placeholders = " OR ".join([f"{c} = ?" for c in LINEUP_COLS])
    match_sql = f"SELECT * FROM barca_matches WHERE {placeholders};"
    matches = pd.read_sql(match_sql, conn, params=[player_id] * len(LINEUP_COLS))
    if matches.empty:
        print("No lineup data for that player.")
        return

    # Barça’s own team ID (most frequent in the table)
    barca_id = matches["home_team_api_id"].mode().iat[0]

    # ------------------------------------------------------------------
    # 1) Most-played opponents
    # ------------------------------------------------------------------
    matches["opponent_id"] = matches.apply(
        lambda r: r["away_team_api_id"] if r["home_team_api_id"] == barca_id else r["home_team_api_id"],
        axis=1,
    )
    opp_counts = matches["opponent_id"].value_counts().head(5)

    if not opp_counts.empty:
        ids = tuple(opp_counts.index)
        # try to fetch pretty names; fall back to raw IDs
        try:
            names = dict(
                conn.execute(
                    f"SELECT team_api_id, COALESCE(team_long_name, team_short_name, team_api_id) "
                    f"FROM team WHERE team_api_id IN ({','.join('?'*len(ids))})",
                    ids,
                )
            )
        except sqlite3.OperationalError:
            names = {i: str(i) for i in ids}

        opp_df = pd.DataFrame(
            {
                "opponent": [names.get(i, str(i)) for i in opp_counts.index],
                "games_played": opp_counts.values,
            }
        )

        fig, ax = plt.subplots()
        opp_df.plot(kind="barh", x="opponent", y="games_played", legend=False, ax=ax)
        ax.set_xlabel("Games Played")
        ax.invert_yaxis()
        ax.set_title(f"{player_name} – most-played opponents")
        plt.show()
    else:
        print("No opponent data to plot.")

    # ---------------------------------------------------------------
    # 2) most-common teammate
    # ---------------------------------------------------------------
    import math, collections
    counts = collections.Counter()

    for _, r in matches.iterrows():
        side_cols = [
            c for c in LINEUP_COLS
            if (r["home_team_api_id"] == barca_id and c.startswith("home_player"))
            or (r["away_team_api_id"] == barca_id and c.startswith("away_player"))
        ]
        lineup = [r[c] for c in side_cols]
        for tm_id in lineup:
            if (
                tm_id is None
                or (isinstance(tm_id, float) and math.isnan(tm_id))
                or tm_id == 0
                or tm_id == player_id
            ):
                continue
            counts[tm_id] += 1

    if counts:
        top_id, top_games = counts.most_common(1)[0]
        print(f"Most-common teammate (ID {top_id}): {top_games} shared matches")
    else:
        print("Couldn’t derive teammate stats.")


    # ------------------------------------------------------------------
    # 3) Peak overall rating
    # ------------------------------------------------------------------
    peak = conn.execute(PEAK_SQL, (player_id,)).fetchone()
    if peak:
        rating, date = peak
        print(f"Peak overall rating: {rating}  (on {date[:10]})")
    else:
        print("No rating history for this player.")

# ------------------------- widget UI --------------------------------
_input = widgets.Text(
    placeholder="Last name (e.g., Messi)",
    description="Player:",
    layout=widgets.Layout(width="260px"),
)
_button = widgets.Button(description="Show")
_button.on_click(lambda _: show_player_cool_stats(_input.value))
display(widgets.HBox([_input, _button]))


For a given team, **show how many goals FC Barcelona scored and conceded**

In [ ]:
# ------------------------------------------------------------
# Barça v Opponent: goals-for & goals-against per season
# ------------------------------------------------------------
import sqlite3, ipywidgets as widgets, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, clear_output

# ---------- helper: list every opponent Barça have faced ----
def _opponents():
    sql = f"""
    select DISTINCT opp.team_long_name as name
    from barca_matches m
        join barca_team opp on opp.team_api_id = m.away_team_api_id
    where m.home_team_api_id = {barca_id}
    union
    select DISTINCT opp.team_long_name
    from barca_matches m
        join barca_team opp on opp.team_api_id = m.home_team_api_id
    where m.away_team_api_id = {barca_id}
    order by name
    """
    return [row[0] for row in conn.execute(sql)]

# ---------- helper: fetch goals-for / against by season ------
def _df_goals_vs(opponent_name: str) -> pd.DataFrame:
    row = conn.execute(
        "select team_api_id from barca_team where team_long_name = ?",
        (opponent_name,)
    ).fetchone()
    if row is None:
        raise ValueError(f"Opponent '{opponent_name}' not found.")
    opp_id = row[0]

    sql = f"""
    select sq.season,
           sum(sq.goals_for)      as goals_for,
           sum(sq.goals_against)  as goals_against
    from (
        select m.season,
               m.home_team_goal as goals_for,
               m.away_team_goal as goals_against
        from barca_matches m
        where m.home_team_api_id = {barca_id}
          and m.away_team_api_id = {opp_id}

        union all

        select m.season,
               m.away_team_goal,
               m.home_team_goal
        from barca_matches m
        where m.away_team_api_id = {barca_id}
          and m.home_team_api_id = {opp_id}
    ) sq
    group by sq.season
    order by sq.season;
    """
    return pd.read_sql_query(sql, conn)

# ---------- plot-builder ------------------------------------
def _plot_barca_vs(opponent_name: str):
    df = _df_goals_vs(opponent_name)
    if df.empty:
        print(f"No matches on record versus {opponent_name}.")
        return

    ax = df.set_index("season")[["goals_for", "goals_against"]].plot(
        kind="bar", figsize=(9, 4)
    )
    ax.set_title(f"Barcelona vs {opponent_name} — goals per season", pad=15)
    ax.set_xlabel("Season")
    ax.set_ylabel("Goals")
    ax.legend(["Goals For", "Goals Against"])
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

# ---------- interactive UI ----------------------------------
def show_barca_vs_opponent():
    opp_dropdown = widgets.Dropdown(
        options=_opponents(),
        description="Opponent:",
        layout=widgets.Layout(width="350px")
    )
    run_btn = widgets.Button(description="Plot", button_style="primary")
    out = widgets.Output()

    def _on_click(_):
        with out:
            clear_output(wait=True)
            _plot_barca_vs(opp_dropdown.value)

    run_btn.on_click(_on_click)
    display(widgets.VBox([opp_dropdown, run_btn, out]))

# call it so the widget appears
show_barca_vs_opponent()


For a given team, **show the results sorted by goal difference**

In [ ]:
# ------------------------------------------------------------
# Barça vs <opponent>: best & worst score-lines
# ------------------------------------------------------------
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ────────────────────────────────────────────────────────────
# helper 1 • list every opponent Barça have faced (for widget)
def _opponents():
    sql = f"""
    select distinct opp.team_long_name
    from barca_matches m
         join barca_team opp on opp.team_api_id = m.away_team_api_id
    where m.home_team_api_id = {barca_id}

    union

    select distinct opp.team_long_name
    from barca_matches m
         join barca_team opp on opp.team_api_id = m.home_team_api_id
    where m.away_team_api_id = {barca_id}

    order by 1
    """
    return [row[0] for row in conn.execute(sql)]

# ────────────────────────────────────────────────────────────
# helper 2 • fetch every match v opponent & compute goal diff
def _df_results(opponent_name: str) -> pd.DataFrame:
    opp_id = conn.execute(
        "select team_api_id from barca_team where team_long_name = ?",
        (opponent_name,)
    ).fetchone()
    if opp_id is None:
        return pd.DataFrame(columns=['date', 'goal_diff'])
    opp_id = opp_id[0]

    sql = f"""
    select date,
           home_team_goal  as hg,
           away_team_goal  as ag,
           home_team_api_id,
           away_team_api_id
    from barca_matches
    where (home_team_api_id = {barca_id} and away_team_api_id = {opp_id})
       or (away_team_api_id = {barca_id} and home_team_api_id = {opp_id})
    order by date
    """
    df = pd.read_sql_query(sql, conn)
    if df.empty:
        return df

    # Barça-perspective goal difference (compute in pandas → no CASE in SQL)
    df['goal_diff'] = np.where(
        df['home_team_api_id'] == barca_id,
        df['hg'] - df['ag'],
        df['ag'] - df['hg']
    )
    df['date'] = pd.to_datetime(df['date']).dt.date.astype(str)
    return df[['date', 'goal_diff']]

# ────────────────────────────────────────────────────────────
# helper 3 • build the bar chart (best ↑   worst ↓)
def _plot_best_worst(opponent_name: str):
    df = _df_results(opponent_name)
    if df.empty:
        print(f"No matches on record versus {opponent_name}.")
        return

    # sort so worst at bottom, best at top
    df = df.sort_values('goal_diff')
    colors = ['crimson' if gd < 0 else 'steelblue' for gd in df['goal_diff']]

    fig, ax = plt.subplots(figsize=(7, max(3, 0.25 * len(df))))
    ax.barh(df['date'], df['goal_diff'], color=colors)
    ax.set_xlabel("Goal Difference  (Barça view)")
    ax.set_title(f"Best ⇢ Worst results vs {opponent_name}", pad=15)
    ax.axvline(0, lw=.8, color='black')
    plt.tight_layout()
    plt.show()

# ────────────────────────────────────────────────────────────
# interactive widget • pick opponent, click “Plot”
def show_best_worst_widget():
    dd   = widgets.Dropdown(options=_opponents(), description="Opponent:")
    btn  = widgets.Button(description="Plot", button_style="primary")
    out  = widgets.Output()

    def _on_click(_):
        with out:
            clear_output(wait=True)
            _plot_best_worst(dd.value)

    btn.on_click(_on_click)
    display(widgets.VBox([dd, btn, out]))

# call once so the widget appears
show_best_worst_widget()

For a given oponent, **show a pie chart with the win/loss/draw ratios**

In [ ]:
# ------------------------------------------------------------
# Barça vs <opponent> • Win / Draw / Loss pie chart
# ------------------------------------------------------------
import sqlite3, pandas as pd, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display, clear_output

# barca_id should already exist (e.g. set earlier with:
# barca_id = conn.execute("select team_api_id from barca_team where team_long_name = 'FC Barcelona'").fetchone()[0])

# ────────────────────────────────────────────────────────────
# helper • list every opponent Barça have faced
def _opponents():
    sql = f"""
    select distinct opp.team_long_name
    from barca_matches m
         join barca_team opp on opp.team_api_id = m.away_team_api_id
    where m.home_team_api_id = {barca_id}

    union

    select distinct opp.team_long_name
    from barca_matches m
         join barca_team opp on opp.team_api_id = m.home_team_api_id
    where m.away_team_api_id = {barca_id}

    order by 1
    """
    return [row[0] for row in conn.execute(sql)]

# ────────────────────────────────────────────────────────────
# helper • pull raw results vs opponent and tally W-D-L
def _wdl_counts(opponent: str):
    opp_id = conn.execute(
        "select team_api_id from barca_team where team_long_name = ?", (opponent,)
    ).fetchone()
    if not opp_id:
        return None
    opp_id = opp_id[0]

    sql = f"""
    select home_team_goal as hg,
           away_team_goal as ag,
           home_team_api_id,
           away_team_api_id
    from barca_matches
    where (home_team_api_id = {barca_id} and away_team_api_id = {opp_id})
       or (away_team_api_id = {barca_id} and home_team_api_id = {opp_id})
    """
    df = pd.read_sql_query(sql, conn)
    if df.empty:
        return None

    # decide result from Barça perspective (no CASE/IF in SQL)
    def _result(row):
        if row['home_team_api_id'] == barca_id:
            return 'Win'  if row['hg'] > row['ag'] else \
                   'Draw' if row['hg'] == row['ag'] else 'Loss'
        else:  # Barça were away
            return 'Win'  if row['ag'] > row['hg'] else \
                   'Draw' if row['ag'] == row['hg'] else 'Loss'

    counts = df.apply(_result, axis=1).value_counts().reindex(['Win', 'Draw', 'Loss'], fill_value=0)
    return counts

# ────────────────────────────────────────────────────────────
# helper • plot a W-D-L pie whose labels include the raw counts
def _plot_pie(opponent: str):
    counts = _wdl_counts(opponent)
    if counts is None:
        print(f"No recorded matches versus {opponent}.")
        return

    labels = [f"{lbl}  ({cnt})" for lbl, cnt in zip(counts.index, counts)]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(counts, labels=labels, startangle=90)   # <- no autopct
    ax.set_title(f"Barça vs {opponent}  •  W / D / L totals", pad=15)
    plt.show()

# ────────────────────────────────────────────────────────────
# interactive widget
def show_wdl_pie_widget():
    dd   = widgets.Dropdown(options=_opponents(), description='Opponent:')
    btn  = widgets.Button(description='Plot', button_style='primary')
    out  = widgets.Output()

    def _on_click(_):
        with out:
            clear_output(wait=True)
            _plot_pie(dd.value)

    btn.on_click(_on_click)
    display(widgets.VBox([dd, btn, out]))

# call once so the widget appears
show_wdl_pie_widget()